# PATSTAT Global - exercise 4
In this exercise we will take a look at the classification tables in PATSTAT Global, and learn how to perform left joins.


In [6]:
# Importing the PATSTAT client
from epo.tipdata.patstat import PatstatClient

# Initialise the PATSTAT client
patstat = PatstatClient(env='PROD') # we use env=PROD in this case

# Access ORM
db = patstat.orm()
# Importing tables as models
from epo.tipdata.patstat.database.models import TLS201_APPLN, TLS224_APPLN_CPC, TLS209_APPLN_IPC


## IPC Classifications

IPC classifications are stored in the `tls209_appln_ipc` table. This table holds information about the IPC classification symbols assigned to patent applications. Important fields in this table include:
- `appln_id`: the unique identifier for the patent application. This field is typically used to link the 209 table (classification) with the `TLS201_APPLN` table. 
- `ipc_class_symbol`: the IPC classification symbol assigned to the application.
- `ipc_version`: the version of the IPC classification system used.
- `ipc_position`: the position of the IPC classification symbol in the classification hierarchy.
- `ipc_gener_auth`: the authority that generated the IPC classification.


### A query about IPC classification with an inner join
Let's build a query aims to retrieve a dataset of South African patent applications filed after 2010, along with their respective IPC classifications and the authority that generated these classifications.

We need to perform a join between `TLS209_APPLN_IPC` and `TLS201_APPLN` tables to link the IPC classification data to the corresponding application data. By joining these tables on the ``appln_id``, we can combine the classification symbols and the authority that generated them with specific details about the patent applications, such as the application authority and number. 

In [7]:
q = db.query(
    TLS201_APPLN.appln_auth,
    TLS201_APPLN.appln_nr,
    TLS209_APPLN_IPC.ipc_class_symbol,
    TLS209_APPLN_IPC.ipc_gener_auth
).join(
    TLS209_APPLN_IPC, TLS209_APPLN_IPC.appln_id == TLS201_APPLN.appln_id, 
).filter(
    TLS201_APPLN.appln_auth == 'ZA',
    TLS201_APPLN.appln_filing_year > 2010
).order_by(
    TLS201_APPLN.appln_nr
)

ipc_inner = patstat.df(q)

ipc_inner


,appln_auth,appln_nr,ipc_class_symbol,ipc_gener_auth
0,ZA,10401D,C07D 309/36,EP
1,ZA,10401D,C07D 213/64,EP
2,ZA,10401D,C07D 277/34,EP
3,ZA,10401D,C07D 271/10,EP
4,ZA,10401D,C07D 311/52,EP
...,...,...,...,...
34876,ZA,928156D,D06P 3/79,EP
34877,ZA,928156D,C08L 39/08,EP
34878,ZA,928156D,C08L 39/06,EP
34879,ZA,928156D,D01F 6/04,EP


### Understanding the results
We can see that typically there are multiple entries in the joined table for a given application. This happens because table `TLS209_APPLN_IPC` has a many-to-many relationship between IPC symbols and applications. 

We also need to consider that the default join in ORM is an inner join. Using an inner join in this query means that only records with matching `appln_id` values in both the `TLS209_APPLN_IPC` and `TLS201_APPLN` tables will be included. This ensures that each resulting record contains both application details and corresponding IPC classification data.

The effect of this inner join is:

- only applications with at least one IPC classification are included,
- applications without an IPC classification are excluded,
- IPC classification entries without a corresponding application are also excluded.


## Left queries in ORM
The implementation of ORM in TIP allows us to perform left queries. If we change the join in the last example to a left join with `isouter=True`, the behavior of the query will change as follows:

Using a left join (`isouter=True`) in this query means that all records from the `TLS201_APPLN` table (the left table) will be included in the result set, regardless of whether there is a matching record in the `TLS209_APPLN_IPC` table (the right table). This ensures that each application detail from the `TLS201_APPLN` table is included, along with the corresponding IPC classification data if it exists.

The effect of this left join is:

- all applications from the `TLS201_APPLN` table will be included, even if they do not have an IPC classification,
- applications without an IPC classification will have `NULL` values for the columns from the `TLS209_APPLN_IPC` table,
- applications with at least one IPC classification will have the corresponding classification data included,
- IPC classification entries without a corresponding application will still be excluded.


In [8]:
q = db.query(
    TLS201_APPLN.appln_nr,
    TLS209_APPLN_IPC.ipc_class_symbol,
).join(
    TLS209_APPLN_IPC, TLS209_APPLN_IPC.appln_id == TLS201_APPLN.appln_id, isouter=True # making it a left join
).filter(
    TLS201_APPLN.appln_auth == 'ZA',
    TLS201_APPLN.appln_filing_year > 2010
).order_by(
    TLS209_APPLN_IPC.ipc_class_symbol
)

ipc_left = patstat.df(q)
print (f'Length of the inner join {len(ipc_inner):,}')
print (f'Length of the left join {len(ipc_left):,}')
ipc_left


Length of the inner join 34,881
Length of the left join 97,919


,appln_nr,ipc_class_symbol
0,703115,None
1,202307523,None
2,202106067,None
3,202301258,None
4,202109543,None
...,...,...
97914,202103822,H10N 60/85
97915,202103822,H10N 69/00
97916,201608040,H10N 97/00
97917,201502852,H10N 97/00


### Filtering the left join using pandas

We can see that the left join indeed gives results where an application has no IPC classification. The size of the ``ipc_left`` dataframe is also bigger than the ``ipc_inner`` dataframe. Let's get only the applications with no IPC classification.

Instead of modifying the query to retrieve only null values, we can filter the existing ``ipc_left`` dataframe using pandas. We use the ``isnull(`` method of pandas to generate a boolean series, where each entry is True if the corresponding ``ipc_class_symbol`` is ``NaN`` and ``False`` otherwise. Using that boolean series, we filter the ``ipc_left`` dataframe to get only the applications without an IPC classification.


In [10]:
# Filtering the left join to get only applications without an IPC classification

# Step 1: create a boolean series where True indicates a missing IPC classification
no_ipc_filter = ipc_left['ipc_class_symbol'].isnull()

# Step 2: filter the left join dataframe using that boolean series
applications_no_ipc = ipc_left[no_ipc_filter]

# Step 3: check lengths to compare
print(f'Length of the inner join: {len(ipc_inner):,}')
print(f'Length of the left join: {len(ipc_left):,}')
print(f'Length of applications without IPC classification: {len(applications_no_ipc):,}')

# Step 4: display the filtered dataframe
applications_no_ipc


Length of the inner join: 34,881
Length of the left join: 97,919
Length of applications without IPC classification: 63,038


,appln_nr,ipc_class_symbol
0,703115,None
1,202307523,None
2,202106067,None
3,202301258,None
4,202109543,None
...,...,...
63033,201707910,None
63034,202204125,None
63035,202200412,None
63036,202203742,None
